In [17]:
#pip install datasets pillow pandas tqdm

In [18]:
!pip install datasets pillow tqdm huggingface_hub

In [19]:
## STEP 1: IMPORTS AND CONFIGURATION

import os
import json
from datasets import load_dataset
from PIL import Image
from collections import defaultdict
from tqdm import tqdm
import random

### 25 Selected Classes with COCO Category IDs

**Vehicles:** car(3), truck(8), bus(6), motorcycle(4), bicycle(2), airplane(5)  
**Person:** person(1)  
**Outdoor:** traffic light(10), stop sign(13), bench(15)  
**Animals:** dog(18), cat(17), horse(19), bird(16), cow(21), elephant(22)  
**Kitchen & Food:** bottle(44), cup(47), bowl(51), pizza(59), cake(61)  
**Furniture:** chair(62), couch(63), bed(65), potted plant(64)

In [20]:
# 25 Selected Classes (CORRECT indices from detection-datasets/coco)

SELECTED_CLASSES = {
    'person': 0,
    'bicycle': 1,
    'car': 2,
    'motorcycle': 3,
    'airplane': 4,
    'bus': 5,
    'train': 6,
    'truck': 7,
    'traffic light': 9,
    'stop sign': 11,
    'bench': 13,
    'bird': 14,
    'cat': 15,
    'dog': 16,
    'horse': 17,
    'cow': 19,
    'elephant': 20,
    'bottle': 39,
    'cup': 41,
    'bowl': 45,
    'pizza': 53,
    'cake': 55,
    'chair': 56,
    'couch': 57,
    'potted plant': 58,
    'bed': 59
}

IMAGES_PER_CLASS = 100
BASE_DIR = "smartvision_dataset"

In [21]:
## STEP 2: LOAD COCO DATASET FROM HUGGING FACE

print("📥 Loading COCO dataset in STREAMING mode (no download)...")
dataset = load_dataset("detection-datasets/coco", split="train", streaming=True)
print("✅ Dataset loaded in streaming mode!")

📥 Loading COCO dataset in STREAMING mode (no download)...


KeyboardInterrupt: 

In [ ]:
## STEP 3: COLLECT IMAGES FROM STREAM

print("\n🔍 Starting image collection from COCO dataset stream...")
print(f"🎯 Target: {IMAGES_PER_CLASS} images per class")
print()

# Initialize storage for collected images
class_images = {class_name: [] for class_name in SELECTED_CLASSES.keys()}
class_counts = {class_name: 0 for class_name in SELECTED_CLASSES.keys()}

# Progress tracking
total_collected = 0
images_processed = 0
max_iterations = 50000  # Safety limit

print("⏳ Processing images from stream...")
print("💡 Progress updates every 100 images collected")
print()

# Iterate through streaming dataset
for idx, item in enumerate(dataset):

    images_processed += 1

    # Progress update every 1000 images processed
    if images_processed % 1000 == 0:
        print(f"📊 Processed {images_processed} images | Collected {total_collected}/{len(SELECTED_CLASSES) * IMAGES_PER_CLASS}")

    # Safety check
    if images_processed >= max_iterations:
        print(f"⚠️ Reached safety limit of {max_iterations} iterations")
        break

    # Check if we have enough images for ALL classes
    if all(count >= IMAGES_PER_CLASS for count in class_counts.values()):
        print("🎉 Successfully collected 100 images for ALL classes!")
        break

    # Get annotations from current image
    annotations = item['objects']
    categories = annotations['category']

    # Check if any of our target classes are in this image
    for cat_id in categories:
        for class_name, class_id in SELECTED_CLASSES.items():
            if cat_id == class_id and class_counts[class_name] < IMAGES_PER_CLASS:

                # Store the ACTUAL image data (not just index!)
                class_images[class_name].append({
                    'image': item['image'],           # PIL Image object
                    'annotations': item['objects'],   # Annotations
                    'idx': images_processed           # For naming
                })

                class_counts[class_name] += 1
                total_collected += 1

                # Progress update every 100 collected
                if total_collected % 100 == 0:
                    print(f"✓ Collected {total_collected}/{len(SELECTED_CLASSES) * IMAGES_PER_CLASS} images")

                break  # Only count once per class

print()
print("="*60)
print("📊 COLLECTION COMPLETE:")
print("="*60)
print(f"Images Processed: {images_processed}")
print(f"Images Collected: {total_collected}")
print()
for class_name, count in sorted(class_counts.items()):
    status = "✅" if count >= IMAGES_PER_CLASS else "⚠️"
    print(f"{status} {class_name:20s}: {count:3d} images")
print("="*60)


🔍 Starting image collection from COCO dataset stream...
🎯 Target: 100 images per class

⏳ Processing images from stream...
💡 Progress updates every 100 images collected

✓ Collected 100/2600 images
✓ Collected 200/2600 images
✓ Collected 300/2600 images
✓ Collected 400/2600 images
✓ Collected 500/2600 images
✓ Collected 600/2600 images
✓ Collected 700/2600 images
✓ Collected 800/2600 images
✓ Collected 900/2600 images
✓ Collected 1000/2600 images
✓ Collected 1100/2600 images
✓ Collected 1200/2600 images
✓ Collected 1300/2600 images
✓ Collected 1400/2600 images
✓ Collected 1500/2600 images
✓ Collected 1600/2600 images
📊 Processed 1000 images | Collected 1682/2600
✓ Collected 1700/2600 images
✓ Collected 1800/2600 images
✓ Collected 1900/2600 images
✓ Collected 2000/2600 images
✓ Collected 2100/2600 images
✓ Collected 2200/2600 images
✓ Collected 2300/2600 images
📊 Processed 2000 images | Collected 2364/2600
✓ Collected 2400/2600 images
✓ Collected 2500/2600 images
📊 Processed 3000 imag

In [ ]:
## STEP 4: CREATE FOLDER STRUCTURE

print("\n📁 Creating project folder structure...")
print()

# Create main directory
os.makedirs(BASE_DIR, exist_ok=True)

# Create subdirectories for Classification task
os.makedirs(f"{BASE_DIR}/classification/train", exist_ok=True)
os.makedirs(f"{BASE_DIR}/classification/val", exist_ok=True)
os.makedirs(f"{BASE_DIR}/classification/test", exist_ok=True)

# Create subdirectories for Detection task
os.makedirs(f"{BASE_DIR}/detection/images", exist_ok=True)
os.makedirs(f"{BASE_DIR}/detection/labels", exist_ok=True)

# Create class folders inside train/val/test
for class_name in SELECTED_CLASSES.keys():
    os.makedirs(f"{BASE_DIR}/classification/train/{class_name}", exist_ok=True)
    os.makedirs(f"{BASE_DIR}/classification/val/{class_name}", exist_ok=True)
    os.makedirs(f"{BASE_DIR}/classification/test/{class_name}", exist_ok=True)

print("✅ Folder structure created successfully!")
print()
print("📂 Structure:")
print(f"""
{BASE_DIR}/
├── classification/
│   ├── train/
│   │   ├── person/
│   │   ├── car/
│   │   └── ... (25 class folders)
│   ├── val/
│   │   └── ... (25 class folders)
│   └── test/
│       └── ... (25 class folders)
│
└── detection/
    ├── images/
    └── labels/
""")


📁 Creating project folder structure...

✅ Folder structure created successfully!

📂 Structure:

smartvision_dataset/
├── classification/
│   ├── train/
│   │   ├── person/
│   │   ├── car/
│   │   └── ... (25 class folders)
│   ├── val/
│   │   └── ... (25 class folders)
│   └── test/
│       └── ... (25 class folders)
│
└── detection/
    ├── images/
    └── labels/



In [ ]:
## STEP 5: TRAIN/VAL/TEST SPLIT (70/15/15)

print("="*70)
print("🔀 Preparing Train/Val/Test splits...")
print("📊 Split Ratio: 70% Train / 15% Val / 15% Test")
print("="*70)
print()

# Initialize metadata dictionary
metadata = {
    'total_images': 0,
    'classes': {},
    'splits': {'train': 0, 'val': 0, 'test': 0}
}

# Create split dictionaries for each class
train_data = {}
val_data = {}
test_data = {}

# Process each class
for class_name in SELECTED_CLASSES.keys():

    all_items = class_images.get(class_name, [])

    if not all_items:
        print(f"⚠️ Warning: No images found for {class_name}")
        continue

    # Calculate split indices
    n = len(all_items)
    train_split = int(0.7 * n)   # 70% for training
    val_split = int(0.85 * n)    # 15% for validation
    # Remaining 15% for test

    # Split the data
    train_data[class_name] = all_items[:train_split]
    val_data[class_name] = all_items[train_split:val_split]
    test_data[class_name] = all_items[val_split:]

    # Store split info in metadata
    metadata['classes'][class_name] = {
        'train': len(train_data[class_name]),
        'val': len(val_data[class_name]),
        'test': len(test_data[class_name]),
        'total': len(all_items)
    }

    metadata['splits']['train'] += len(train_data[class_name])
    metadata['splits']['val'] += len(val_data[class_name])
    metadata['splits']['test'] += len(test_data[class_name])
    metadata['total_images'] += len(all_items)

    print(f"{class_name:20s}: Train={len(train_data[class_name]):3d} | Val={len(val_data[class_name]):2d} | Test={len(test_data[class_name]):2d}")

🔀 Preparing Train/Val/Test splits...
📊 Split Ratio: 70% Train / 15% Val / 15% Test

person              : Train= 70 | Val=15 | Test=15
bicycle             : Train= 70 | Val=15 | Test=15
car                 : Train= 70 | Val=15 | Test=15
motorcycle          : Train= 70 | Val=15 | Test=15
airplane            : Train= 70 | Val=15 | Test=15
bus                 : Train= 70 | Val=15 | Test=15
train               : Train= 70 | Val=15 | Test=15
truck               : Train= 70 | Val=15 | Test=15
traffic light       : Train= 70 | Val=15 | Test=15
stop sign           : Train= 70 | Val=15 | Test=15
bench               : Train= 70 | Val=15 | Test=15
bird                : Train= 70 | Val=15 | Test=15
cat                 : Train= 70 | Val=15 | Test=15
dog                 : Train= 70 | Val=15 | Test=15
horse               : Train= 70 | Val=15 | Test=15
cow                 : Train= 70 | Val=15 | Test=15
elephant            : Train= 70 | Val=15 | Test=15
bottle              : Train= 70 | Val=15 | Test=1

In [ ]:
import os
from PIL import Image
from tqdm import tqdm
import json

print("="*70)
print("💾 STEP 6: SAVING IMAGES TO DISK")
print("="*70)
print()

# PART A: SAVE CLASSIFICATION IMAGES


print("📁 PART A: Saving Classification Images...")
print("   Format: Cropped objects, 224x224 pixels\n")

classification_stats = {'train': 0, 'val': 0, 'test': 0}

# Process each split
for split_name, split_data in [('train', train_data), ('val', val_data), ('test', test_data)]:

    print(f"📂 Processing {split_name.upper()} split...")

    # Process each class
    for class_name, items in tqdm(split_data.items(), desc=f"  {split_name}"):

        class_folder = f"{BASE_DIR}/classification/{split_name}/{class_name}"

        # Save each image
        for img_idx, item in enumerate(items):

            img = item['image']
            annotations = item['annotations']
            bboxes = annotations['bbox']
            categories = annotations['category']

            class_id = SELECTED_CLASSES[class_name]

            # Find bbox for this class
            for bbox, cat_id in zip(bboxes, categories):
                if cat_id == class_id:
                    x, y, w, h = bbox

                    try:
                        # Crop and resize
                        cropped_img = img.crop((x, y, x + w, y + h))
                        cropped_img = cropped_img.resize((224, 224), Image.LANCZOS)

                        # Save
                        img_filename = f"{class_name}_{split_name}_{img_idx:04d}.jpg"
                        img_path = os.path.join(class_folder, img_filename)
                        cropped_img.save(img_path, quality=95)

                        classification_stats[split_name] += 1

                    except Exception as e:
                        print(f"⚠️ Error: {class_name} image {img_idx}: {e}")

                    break

print()
print("="*70)
print("✅ CLASSIFICATION IMAGES SAVED!")
print("="*70)
print(f"📊 Train: {classification_stats['train']} images")
print(f"📊 Val:   {classification_stats['val']} images")
print(f"📊 Test:  {classification_stats['test']} images")
print(f"📊 Total: {sum(classification_stats.values())} images")
print()

💾 STEP 6: SAVING IMAGES TO DISK

📁 PART A: Saving Classification Images...
   Format: Cropped objects, 224x224 pixels

📂 Processing TRAIN split...


  train: 100%|██████████| 26/26 [00:27<00:00,  1.06s/it]


📂 Processing VAL split...


  val: 100%|██████████| 26/26 [00:05<00:00,  4.89it/s]


📂 Processing TEST split...


  test: 100%|██████████| 26/26 [00:05<00:00,  4.65it/s]


✅ CLASSIFICATION IMAGES SAVED!
📊 Train: 1820 images
📊 Val:   390 images
📊 Test:  390 images
📊 Total: 2600 images



In [ ]:
# PART B: SAVE DETECTION IMAGES (YOLO FORMAT)

print("="*70)
print("📁 PART B: Saving Detection Images & Annotations...")
print("   Format: Full images with YOLO .txt labels\n")

detection_stats = {'images': 0, 'annotations': 0, 'objects': 0}

# COCO to YOLO class mapping
coco_to_yolo = {class_id: idx for idx, class_id in enumerate(SELECTED_CLASSES.values())}

# Combine train + val for detection
all_detection_data = []
for class_name in SELECTED_CLASSES.keys():
    all_detection_data.extend(train_data.get(class_name, []))
    all_detection_data.extend(val_data.get(class_name, []))

print(f"📊 Total detection images: {len(all_detection_data)}\n")

# Save images and create YOLO labels
for img_idx, item in enumerate(tqdm(all_detection_data, desc="Saving detection data")):

    img = item['image']
    img_width, img_height = img.size

    # Save full image
    img_filename = f"image_{img_idx:06d}.jpg"
    img_path = os.path.join(f"{BASE_DIR}/detection/images", img_filename)
    img.save(img_path, quality=95)
    detection_stats['images'] += 1

    # Get annotations
    annotations = item['annotations']
    bboxes = annotations['bbox']
    categories = annotations['category']

    # Create YOLO annotation
    label_filename = f"image_{img_idx:06d}.txt"
    label_path = os.path.join(f"{BASE_DIR}/detection/labels", label_filename)

    yolo_annotations = []
    objects_count = 0

    for bbox, cat_id in zip(bboxes, categories):
        if cat_id in coco_to_yolo:
            x, y, w, h = bbox

            # Convert to YOLO format (normalized)
            x_center = (x + w/2) / img_width
            y_center = (y + h/2) / img_height
            w_norm = w / img_width
            h_norm = h / img_height

            yolo_class_id = coco_to_yolo[cat_id]
            yolo_line = f"{yolo_class_id} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}"
            yolo_annotations.append(yolo_line)
            objects_count += 1

    # Save label file
    if yolo_annotations:
        with open(label_path, 'w') as f:
            f.write('\n'.join(yolo_annotations))
        detection_stats['annotations'] += 1
        detection_stats['objects'] += objects_count

print()
print("="*70)
print("✅ DETECTION DATASET CREATED!")
print("="*70)
print(f"📊 Images:     {detection_stats['images']}")
print(f"📊 Labels:     {detection_stats['annotations']}")
print(f"📊 Objects:    {detection_stats['objects']}")
print(f"📊 Avg/image:  {detection_stats['objects']/detection_stats['images']:.2f}")
print()

📁 PART B: Saving Detection Images & Annotations...
   Format: Full images with YOLO .txt labels

📊 Total detection images: 2210



Saving detection data: 100%|██████████| 2210/2210 [00:30<00:00, 71.44it/s]


✅ DETECTION DATASET CREATED!
📊 Images:     2210
📊 Labels:     2210
📊 Objects:    22163
📊 Avg/image:  10.03



In [ ]:
# PART C: CREATE YOLO CONFIG FILE

print("📝 Creating YOLO configuration file...\n")

yaml_content = f"""# SmartVision Dataset - YOLOv8 Configuration
path: {os.path.abspath(BASE_DIR)}/detection
train: images
val: images

names:
  0: person
  1: bicycle
  2: car
  3: motorcycle
  4: airplane
  5: bus
  6: train
  7: truck
  8: traffic light
  9: stop sign
  10: bench
  11: bird
  12: cat
  13: dog
  14: horse
  15: cow
  16: elephant
  17: bottle
  18: cup
  19: bowl
  20: pizza
  21: cake
  22: chair
  23: couch
  24: potted plant
  25: bed

nc: 26
"""

yaml_path = f"{BASE_DIR}/detection/data.yaml"
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(f"✅ Created: {yaml_path}\n")

📝 Creating YOLO configuration file...

✅ Created: smartvision_dataset/detection/data.yaml



In [ ]:
# PART D: SAVE METADATA

print("📊 Saving metadata...\n")

metadata['classification'] = classification_stats
metadata['detection'] = detection_stats
metadata['dataset_path'] = os.path.abspath(BASE_DIR)

metadata_path = f"{BASE_DIR}/dataset_metadata.json"
with open(metadata_path, 'w') as f:
    json.dump(metadata, indent=2, fp=f)

print(f"✅ Saved: {metadata_path}\n")

📊 Saving metadata...

✅ Saved: smartvision_dataset/dataset_metadata.json



In [ ]:
print("="*70)
print("🎉 DATASET SETUP COMPLETE!")
print("="*70)
print()
print(f"📁 Location: {os.path.abspath(BASE_DIR)}")
print()
print("📂 Classification Dataset:")
print(f"   ├─ Train:  {classification_stats['train']} images (70%)")
print(f"   ├─ Val:    {classification_stats['val']} images (15%)")
print(f"   ├─ Test:   {classification_stats['test']} images (15%)")
print(f"   └─ Total:  {sum(classification_stats.values())} cropped images (224x224)")
print()
print("📂 Detection Dataset:")
print(f"   ├─ Images: {detection_stats['images']} full images")
print(f"   ├─ Labels: {detection_stats['annotations']} YOLO .txt files")
print(f"   └─ Objects: {detection_stats['objects']} annotated objects")
print()
print("="*70)
print("✅ LEARNERS CAN NOW START:")
print("="*70)
print("Step 7:  Exploratory Data Analysis (EDA)")
print("Step 8:  Train Classification Models")
print("Step 9:  Train YOLO Detection Model")
print("Step 10: Build Streamlit Application")
print("Step 11: Deploy to Hugging Face Spaces")
print("="*70)

🎉 DATASET SETUP COMPLETE!

📁 Location: d:\Smart vision\smartvision_dataset

📂 Classification Dataset:
   ├─ Train:  1820 images (70%)
   ├─ Val:    390 images (15%)
   ├─ Test:   390 images (15%)
   └─ Total:  2600 cropped images (224x224)

📂 Detection Dataset:
   ├─ Images: 2210 full images
   ├─ Labels: 2210 YOLO .txt files
   └─ Objects: 22163 annotated objects

✅ LEARNERS CAN NOW START:
Step 7:  Exploratory Data Analysis (EDA)
Step 8:  Train Classification Models
Step 9:  Train YOLO Detection Model
Step 10: Build Streamlit Application
Step 11: Deploy to Hugging Face Spaces


# my yolo

In [ ]:
import os
import random
from tqdm import tqdm

print("="*70)
print("📁 PART B: Saving Split Detection Images & Annotations...")
print("   Format: Strict YOLOv8 Train/Val Directory Split\n")

# 1. Establish Structured YOLOv8 Directories
BASE_DIR = "smartvision_dataset"
DET_DIR = os.path.join(BASE_DIR, "detection")

splits = ['train', 'val']
subdirs = ['images', 'labels']

for s in splits:
    for sd in subdirs:
        os.makedirs(os.path.join(DET_DIR, s, sd), exist_ok=True)

detection_stats = {
    'train_images': 0, 'train_objects': 0,
    'val_images': 0, 'val_objects': 0
}

# COCO to YOLO class mapping
coco_to_yolo = {class_id: idx for idx, class_id in enumerate(SELECTED_CLASSES.values())}

# Combine dataset elements
all_detection_data = []
for class_name in SELECTED_CLASSES.keys():
    all_detection_data.extend(train_data.get(class_name, []))
    all_detection_data.extend(val_data.get(class_name, []))

# Shuffle data to ensure balanced class distributions across splits
random.seed(42)
random.shuffle(all_detection_data)

# Calculate 80/20 train/validation index cutoff boundary
split_idx = int(len(all_detection_data) * 0.8)

print(f"📊 Total source images: {len(all_detection_data)}")
print(f"📊 Allocating -> Train: {split_idx} images | Val: {len(all_detection_data) - split_idx} images\n")

# 2. Save Images and Create YOLO Labels Loop
for img_idx, item in enumerate(tqdm(all_detection_data, desc="Processing YOLO Dataset")):
    
    # Determine split destination dynamically
    current_split = 'train' if img_idx < split_idx else 'val'
    
    img = item['image']
    img_width, img_height = img.size

    # Save full image to target split folder
    img_filename = f"image_{img_idx:06d}.jpg"
    img_path = os.path.join(DET_DIR, current_split, "images", img_filename)
    img.save(img_path, quality=95)
    detection_stats[f'{current_split}_images'] += 1

    # Extract framing parameters
    annotations = item['annotations']
    bboxes = annotations['bbox']
    categories = annotations['category']

    # Set label destination
    label_filename = f"image_{img_idx:06d}.txt"
    label_path = os.path.join(DET_DIR, current_split, "labels", label_filename)

    yolo_annotations = []
    for bbox, cat_id in zip(bboxes, categories):
        if cat_id in coco_to_yolo:
            x, y, w, h = bbox

            # Normalized conversion algorithm -> YOLO standard [0, 1]
            x_center = (x + w / 2) / img_width
            y_center = (y + h / 2) / img_height
            w_norm = w / img_width
            h_norm = h / img_height

            yolo_class_id = coco_to_yolo[cat_id]
            yolo_line = f"{yolo_class_id} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}"
            yolo_annotations.append(yolo_line)
            detection_stats[f'{current_split}_objects'] += 1

    # Save label file if target bounding boxes exist
    if yolo_annotations:
        with open(label_path, 'w') as f:
            f.write('\n'.join(yolo_annotations))
    else:
        # Create an empty annotation file for images containing background negatives
        open(label_path, 'w').close()

# 3. Automatically generate the critical configuration YAML file for YOLOv8
yaml_content = f"""
path: {os.path.abspath(DET_DIR)}
train: train/images
val: val/images

names:
"""
for idx, name in enumerate(sorted(SELECTED_CLASSES.keys())):
    yaml_content += f"  {idx}: {name}\n"

yaml_path = os.path.join(BASE_DIR, "dataset.yaml")
with open(yaml_path, 'w') as f:
    f.write(yaml_content.strip())

print("\n" + "="*70)
print("✅ COMPLIANT YOLOV8 DATASET ARCHITECTURE CREATED!")
print("="*70)
print(f"📁 Train Split:  {detection_stats['train_images']} images | {detection_stats['train_objects']} object instances")
print(f"📁 Val Split:    {detection_stats['val_images']} images | {detection_stats['val_objects']} object instances")
print(f"📄 Generated Config File: {yaml_path}")
print("="*70)

📁 PART B: Saving Split Detection Images & Annotations...
   Format: Strict YOLOv8 Train/Val Directory Split

📊 Total source images: 2210
📊 Allocating -> Train: 1768 images | Val: 442 images



Processing YOLO Dataset: 100%|██████████| 2210/2210 [00:27<00:00, 81.80it/s] 


✅ COMPLIANT YOLOV8 DATASET ARCHITECTURE CREATED!
📁 Train Split:  1768 images | 17838 object instances
📁 Val Split:    442 images | 4325 object instances
📄 Generated Config File: smartvision_dataset\dataset.yaml


In [ ]:
from ultralytics import YOLO
import os

# 1. Load baseline pre-trained architecture weights
model = YOLO('yolov8n.pt') 

print("🏋️ Launching YOLOv8 Custom Object Detection Training Pipeline...")
results = model.train(
    data=os.path.abspath("smartvision_dataset/dataset.yaml"), 
    epochs=10,                                                # Adjusted to 10 for rapid local execution testing
    imgsz=640,                                               
    batch=16,                                                
    name='yolov8n_custom'                                    
)

print("🎉 Training complete! Local weights generated successfully.")

🏋️ Launching YOLOv8 Custom Object Detection Training Pipeline...
Ultralytics 8.4.51  Python-3.13.11 torch-2.12.0+cpu CPU (Intel Core i3-1005G1 1.20GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=d:\Smart vision\smartvision_dataset\dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_custom-

# Step 1.2: Exploratory Data Analysis (EDA)

# Analyze Class Distribution

In [ ]:
def new_func():
    %pip install ultralytics matplotlib pandas torch torchvision pillow seaborn

new_func()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
import json # Added for loading metadata

# Ensure BASE_DIR is defined
BASE_DIR = "smartvision_dataset"

# Check if metadata is already defined; if not, load it from file
if 'metadata' not in globals():
    METADATA_PATH = os.path.join(BASE_DIR, "dataset_metadata.json")
    try:
        with open(METADATA_PATH, 'r') as f:
            metadata = json.load(f)
        print(f"✅ Loaded metadata from {METADATA_PATH}")
    except FileNotFoundError:
        print(f"❌ Error: {METADATA_PATH} not found. Please ensure data preparation steps are completed and `dataset_metadata.json` is generated.")
        # If metadata is critical and not found, stop further execution
        raise
    except json.JSONDecodeError:
        print(f"❌ Error: Could not decode JSON from {METADATA_PATH}. File might be corrupted.")
        raise
    except Exception as e:
        print(f"❌ An unexpected error occurred while loading metadata: {e}")
        raise

# 1. Extract the class distribution data from your metadata dictionary
# We want to see the total images per class (should be 100 each)
class_data = []
for class_name, info in metadata['classes'].items():
    class_data.append({
        'Class Name': class_name,
        'Total Images': info['total']
    })

# 2. Convert to DataFrame
df_plot = pd.DataFrame(class_data)

# 3. Create the Visualization
plt.figure(figsize=(15, 6))
sns.barplot(data=df_plot, x='Class Name', y='Total Images', hue='Class Name', palette='viridis', legend=False)

# 4. Add details for clarity
plt.title("SmartVision AI: Class Distribution Verification (EDA)", fontsize=15)
plt.axhline(y=100, color='r', linestyle='--', label='Target: 100 images') # Target line
plt.xticks(rotation=45)
plt.ylabel("Number of Images")
plt.xlabel("Category")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# 5. Print Split Summary
print("\n" + "="*30)
print("📦 FINAL DATASET SUMMARY")
print("="*30)
print(f"Total Images: {metadata['total_images']}")
print(f"Train Split:  {metadata['splits']['train']}")
print(f"Val Split:    {metadata['splits']['val']}")
print(f"Test Split:   {metadata['splits']['test']}")
print("="*30)

ModuleNotFoundError: No module named 'matplotlib'

# B. Examine Image Resolutions

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Extract width and height from the collected PIL images
resolutions = []

for class_name, items in class_images.items():
    for item in items:
        # 'image' is the PIL object we stored during collection
        img = item['image']
        resolutions.append({
            'Width': img.width,
            'Height': img.height,
            'Class': class_name
        })

# 2. Create the DataFrame
res_df = pd.DataFrame(resolutions)

# 3. Create the Scatter Plot
plt.figure(figsize=(10, 6))
plt.scatter(res_df['Width'], res_df['Height'], alpha=0.4, c='blue', edgecolors='none')

# 4. Add labels and formatting
plt.title("SmartVision AI: Image Resolution Distribution", fontsize=14)
plt.xlabel("Width (pixels)")
plt.ylabel("Height (pixels)")
plt.grid(True, linestyle='--', alpha=0.6)

# Optional: Add a line showing the 224x224 target size for Phase 2
plt.axvline(x=224, color='r', linestyle=':', label='Target Resize (224px)')
plt.axhline(y=224, color='r', linestyle=':')
plt.legend()

plt.show()

# 5. Print Resolution Statistics
print("\n📊 Resolution Summary Statistics:")
print(res_df[['Width', 'Height']].describe().loc[['min', 'max', 'mean']])

ModuleNotFoundError: No module named 'pandas'

# # Visualize Samples with Annotations

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def plot_smartvision_sample(class_name, image_index=0):
    # 1. Get the data from your collection
    sample_item = class_images[class_name][image_index]
    image = sample_item['image']
    obj_data = sample_item['annotations'] # This is a dict of lists

    fig, ax = plt.subplots(1, figsize=(10, 7))
    ax.imshow(image)

    # 2. Iterate through the objects using zip
    # item['objects'] has keys: 'bbox', 'category', 'id', etc.
    for bbox, cat_id in zip(obj_data['bbox'], obj_data['category']):

        # 3. Only draw boxes for the classes we care about
        # We find the name matching the ID from our SELECTED_CLASSES
        name = [k for k, v in SELECTED_CLASSES.items() if v == cat_id]
        label = name[0] if name else "unknown"

        # 4. Draw the bounding box (COCO format: x, y, width, height)
        x, y, w, h = bbox
        rect = patches.Rectangle((x, y), w, h, linewidth=2,
                                 edgecolor='lime', facecolor='none')
        ax.add_patch(rect)

        # 5. Add label text
        plt.text(x, y, label, color='black', fontweight='bold',
                 bbox=dict(facecolor='lime', alpha=0.8, pad=2))

    plt.title(f"EDA: Sample Image for Class '{class_name}'", fontsize=14)
    plt.axis('off')
    plt.show()

# Visualize a specific example, e.g., the first 'car' image
plot_smartvision_sample('car', 0)

ModuleNotFoundError: No module named 'matplotlib'